# 广义相对论张量计算可视化

本 Notebook 利用 **SymPy** 进行符号计算，展示广义相对论中关键张量的逐分量可视化。

**内容**
1. 基本工具函数（Christoffel 符号、Riemann 张量、Ricci 张量、Ricci 标量、Einstein 张量）
2. 示例一：平直时空（Minkowski 度规）
3. 示例二：二维球面（$S^2$，Ricci 标量验证）
4. 示例三：Schwarzschild 度规（真空解，$R=0$ 验证）


In [ ]:
import sympy as sp
from sympy import symbols, sin, cos, diag, simplify, diff, latex, Matrix, Integer, pi
sp.init_printing(use_unicode=True)
print('SymPy version:', sp.__version__)

## 1  工具函数

### 1.1  Christoffel 符号 $\Gamma^\lambda_{\mu\nu}$

$$\Gamma^\lambda_{\mu\nu} = \frac{1}{2}\,g^{\lambda\sigma}\bigl(\partial_\mu g_{\sigma\nu}
+ \partial_\nu g_{\sigma\mu} - \partial_\sigma g_{\mu\nu}\bigr)$$

In [ ]:
def christoffel(g, coords):
    """返回 Christoffel 符号 Γ[λ][μ][ν] = Gamma^λ_μν"""
    n = len(coords)
    g_inv = g.inv()
    G = [[[Integer(0)]*n for _ in range(n)] for _ in range(n)]
    for lam in range(n):
        for mu in range(n):
            for nu in range(n):
                s = Integer(0)
                for sig in range(n):
                    s += g_inv[lam, sig] * (
                        diff(g[sig, nu], coords[mu])
                        + diff(g[sig, mu], coords[nu])
                        - diff(g[mu, nu], coords[sig])
                    )
                G[lam][mu][nu] = simplify(s / 2)
    return G


def print_christoffel(G, coords, label=''):
    """打印所有非零 Christoffel 符号"""
    coord_names = [str(x) for x in coords]
    n = len(coords)
    print(f'Christoffel 符号 {label}')
    found = False
    for lam in range(n):
        for mu in range(n):
            for nu in range(mu, n):
                val = G[lam][mu][nu]
                if val != 0:
                    print(f'  Γ^{coord_names[lam]}_{{{coord_names[mu]}{coord_names[nu]}}} = {val}')
                    found = True
    if not found:
        print('  (所有分量均为零 — 平直时空)')

### 1.2  Riemann 曲率张量 $R^a{}_{bcd}$

$$R^a{}_{bcd} = \partial_c\Gamma^a_{db} - \partial_d\Gamma^a_{cb}
+ \Gamma^a_{ce}\Gamma^e_{db} - \Gamma^a_{de}\Gamma^e_{cb}$$

In [ ]:
def riemann(G, coords):
    """返回 Riemann 张量 R[a][b][c][d] = R^a_bcd"""
    n = len(coords)
    R = [[[[Integer(0)]*n for _ in range(n)] for _ in range(n)] for _ in range(n)]
    for a in range(n):
        for b in range(n):
            for c in range(n):
                for d in range(n):
                    t1 = diff(G[a][d][b], coords[c])
                    t2 = diff(G[a][c][b], coords[d])
                    t3 = sum(G[a][c][e]*G[e][d][b] for e in range(n))
                    t4 = sum(G[a][d][e]*G[e][c][b] for e in range(n))
                    R[a][b][c][d] = simplify(t1 - t2 + t3 - t4)
    return R


def print_riemann(R, coords, label=''):
    """打印所有非零 Riemann 分量"""
    coord_names = [str(x) for x in coords]
    n = len(coords)
    print(f'Riemann 张量 {label}')
    found = False
    for a in range(n):
        for b in range(n):
            for c in range(n):
                for d in range(n):
                    val = R[a][b][c][d]
                    if val != 0:
                        print(f'  R^{coord_names[a]}_{{{coord_names[b]}{coord_names[c]}{coord_names[d]}}} = {val}')
                        found = True
    if not found:
        print('  (所有分量均为零 — 平直时空)')

### 1.3  Ricci 张量 $R_{ab}$、Ricci 标量 $R$ 与 Einstein 张量 $G_{ab}$

$$R_{ab} = R^c{}_{acb}, \quad R = g^{ab}R_{ab}, \quad G_{ab} = R_{ab} - \tfrac{1}{2}g_{ab}R$$

In [ ]:
def ricci_tensor(R, n):
    """R_ab = R^c_acb"""
    Ric = Matrix([[simplify(sum(R[c][a][c][b] for c in range(n)))
                  for b in range(n)] for a in range(n)])
    return Ric


def ricci_scalar(Ric, g_inv):
    return simplify((g_inv * Ric).trace())


def einstein_tensor(Ric, R_sc, g):
    return simplify(Ric - sp.Rational(1, 2) * R_sc * g)


def show_tensor(T, label):
    """以矩阵形式展示二阶张量"""
    print(f'{label} =')
    sp.pprint(T, use_unicode=True)
    print()

## 2  示例一：Minkowski 平直时空

度规 $g_{\mu\nu} = \mathrm{diag}(-1,+1,+1,+1)$，坐标 $(t, x, y, z)$。
平直时空的所有曲率张量应为零。

In [ ]:
t, x, y, z = symbols('t x y z')
coords_mink = [t, x, y, z]

g_mink = diag(-1, 1, 1, 1)
print('Minkowski 度规 g_μν =')
sp.pprint(g_mink)
print()

In [ ]:
G_mink = christoffel(g_mink, coords_mink)
print_christoffel(G_mink, coords_mink, '(Minkowski)')

In [ ]:
R_mink  = riemann(G_mink, coords_mink)
Ric_mink = ricci_tensor(R_mink, 4)
Rsc_mink = ricci_scalar(Ric_mink, g_mink.inv())
print_riemann(R_mink, coords_mink, '(Minkowski)')
print('Ricci 标量 R =', Rsc_mink)

## 3  示例二：二维球面 $S^2$

度规 $g_{ij} = \mathrm{diag}(a^2,\;a^2\sin^2\theta)$，坐标 $(\theta,\varphi)$。
对于半径为 $a$ 的球面，Ricci 标量应为 $R = 2/a^2$。

In [ ]:
theta, phi_sym, a = symbols('theta phi a', positive=True)
coords_s2 = [theta, phi_sym]

g_s2 = Matrix([[a**2, 0],
               [0, a**2 * sin(theta)**2]])

print('S² 度规 g_ij =')
sp.pprint(g_s2)
print()

In [ ]:
G_s2 = christoffel(g_s2, coords_s2)
print_christoffel(G_s2, coords_s2, '(S²)')

In [ ]:
R_s2   = riemann(G_s2, coords_s2)
Ric_s2 = ricci_tensor(R_s2, 2)
Rsc_s2 = ricci_scalar(Ric_s2, g_s2.inv())

print_riemann(R_s2, coords_s2, '(S²)')
print()
show_tensor(Ric_s2, 'Ricci 张量 R_ij')
print('Ricci 标量 R =', Rsc_s2, '  (= 2/a² ✓)')

## 4  示例三：Schwarzschild 度规

$$ds^2 = -\!\left(1-\frac{r_s}{r}\right)c^2 dt^2
+ \left(1-\frac{r_s}{r}\right)^{-1}\!dr^2
+ r^2 d\theta^2 + r^2\sin^2\theta\,d\varphi^2$$

其中 $r_s = 2GM/c^2$ 为 Schwarzschild 半径，坐标 $(t, r, \theta, \varphi)$，
本例令 $c=1$（自然单位）。
作为真空解，Ricci 张量与 Ricci 标量均为零：$R_{\mu\nu}=0$，$R=0$。

In [ ]:
t_s, r_s_coord, th_s, ph_s = symbols('t r theta phi')
rs = symbols('r_s', positive=True)   # Schwarzschild 半径
coords_schw = [t_s, r_s_coord, th_s, ph_s]

f = 1 - rs / r_s_coord
g_schw = diag(-f, 1/f, r_s_coord**2, r_s_coord**2 * sin(th_s)**2)

print('Schwarzschild 度规 g_μν =')
sp.pprint(g_schw)
print()

In [ ]:
G_schw = christoffel(g_schw, coords_schw)
print_christoffel(G_schw, coords_schw, '(Schwarzschild)')

In [ ]:
R_schw   = riemann(G_schw, coords_schw)
Ric_schw = ricci_tensor(R_schw, 4)
Rsc_schw = ricci_scalar(Ric_schw, g_schw.inv())

show_tensor(Ric_schw, 'Ricci 张量 R_μν (Schwarzschild)')
print('Ricci 标量 R =', Rsc_schw, '  (应为 0 ✓)')

In [ ]:
G_ein = einstein_tensor(Ric_schw, Rsc_schw, g_schw)
show_tensor(G_ein, 'Einstein 张量 G_μν (Schwarzschild)')

### 4.1  Riemann 张量的独立非零分量（非零曲率体现黑洞引力场）

In [ ]:
print_riemann(R_schw, coords_schw, '(Schwarzschild)')